#Telecom Domain Read & Write Ops Assignment - Building Datalake & Lakehouse
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
Create catalog if not exists telecom_catalog_assign;
Create schema if not exists telecom_catalog_assign.landing_zone;
Create Volume if not exists telecom_catalog_assign.landing_zone.landing_vol;




In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

##2. Filesystem operations
1. Write dbutils.fs code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:

dbutils.fs.cp("/Workspace/Users/skvhari456@gmail.com/Inceptez/tower_logs_region1.txt","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")
dbutils.fs.cp("/Workspace/Users/skvhari456@gmail.com/Inceptez/tower_logs_region2.txt","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")

##3. Spark Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#rdd=spark.read.option("sep","|").option("header","true").option("inferSchema","true").option("recursiveFileLookup","true").option("pathGlobFilter","*region*").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/")
#rdd=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/",pathGlobFilter="*region*",sep="|",header="true",recursiveFileLookup="True")
rdd=spark.read.format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/",pathGlobFilter="*region*",sep="|",header="true",recursiveFileLookup="True")

display(rdd)
print(rdd.printSchema())

##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
rd=spark.read.option("inferSchema","false").option("header","true").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
display(rd)
print(rd.printSchema())


##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
display(spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/").toDF("ID","Name","Age","City","PLAN"))

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
structure=StructType([StructField("ID",IntegerType(),True),StructField("Name",StringType(),True),StructField("Age",IntegerType(),True),StructField("City",StringType(),True),StructField("PLAN",StringType(),True)])
rd2=spark.read.option("inferSchema","true").schema(structure).format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
display(rd2)
print(rd2.printSchema())

In [0]:
struct="Id int,Name string,Age int,City string,PLAN string"
rd3=spark.read.schema(struct).format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
display(rd3)
print(rd3.printSchema())

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.mode("overwrite").insertInto("catalog1_dropme.schema1_dropme.vijaytable")

display(spark.read.table("catalog1_dropme.schema1_dropme.vijaytable"))

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.option("compression","snappy").option("rowTag","ROW").mode("overwrite").format("xml").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme")

display(spark.read.option("rowTag","ROW").format("xml").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme"))

In [0]:
%sql
Create Table if not exists catalog1_dropme.schema1_dropme.VijayTable(ID integer,FName string,LName string,Age int,Profession string)


In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()


s.write.mode("overwrite").saveAsTable("catalog1_dropme.schema1_dropme.VijayTable")

display(spark.read.table("catalog1_dropme.schema1_dropme.VijayTable"))

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.option("compression","snappy").mode("overwrite").format("delta").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/deltadirectory/")

spark.read.format("delta").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/deltadirectory/").explain()

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.option("compression","snappy").mode("overwrite").format("parquet").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme")

display(spark.read.format("parquet").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme"))

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.option("compression","snappy").mode("overwrite").format("json").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme")

display(spark.read.format("json").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme"))


In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

structure=StructType([StructField("Id",IntegerType(),True),StructField("FName",StringType(),True),StructField("LName",StringType(),True),
                      StructField("Age",IntegerType(),True),StructField("Profession",StringType(),True)])
s=spark.read.schema(structure).option("InferSchema",True).format("csv").load("/Workspace/Users/skvhari456@gmail.com/Inceptez/custs")
s.show(2)
s.printSchema()
s.write.mode("overwrite").format("orc").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme")

display(spark.read.format("orc").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme"))

##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

##15. Do a final exercise of defining one/two liner of... 
1. When to use/benifits csv
2. When to use/benifits json
3. When to use/benifit orc
4. When to use/benifit parquet
5. When to use/benifit delta
6. When to use/benifit xml
7. When to use/benifit delta tables
